In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q04/annotated/checkpoints/pre_cell_2.pickle

trying: ['orders']


me:  2
trying: ['load_lineitem']
me:  1
trying: ['lineitem']


me:  1
trying: ['tpch_parent']
me:  0
trying: ['pd']
me:  0
trying: ['load_orders']
me:  2
trying: ['STORAGE_OPTS']
me:  0
trying: ['DATA_ROOT']
me:  0


Declaring variable tpch_parent
Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable orders
Declaring variable load_orders


In [4]:
%%RecordEvent
%%time
### cell 2 ###

# Optimized for cudf
date_start = "1993-08-01"
date_end = "1993-11-01"

# Filter lineitem on GPU
flineitem = lineitem[lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE]

# Filter orders on GPU using string literals to avoid pd.Timestamp
forders = orders[(orders.O_ORDERDATE >= date_start) & (orders.O_ORDERDATE < date_end)]

# Use GPU‐accelerated merge instead of isin to reduce CPU work
keys = flineitem[["L_ORDERKEY"]].drop_duplicates()
jn = forders.merge(keys, left_on="O_ORDERKEY", right_on="L_ORDERKEY", how="inner")

# Group and count on GPU
total = jn.groupby("O_ORDERPRIORITY", as_index=False)["O_ORDERKEY"].count()

CPU times: user 71.4 ms, sys: 45.3 ms, total: 117 ms
Wall time: 137 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_q04/checkpoints/post_cell_2_try_4.pickle

migration speed (bps): 765412158.5253456
---------------------------
variables to migrate:
keys 48
lineitem 48
date_end 59
load_orders 144
total 48
pd 72
date_start 59
STORAGE_OPTS 64
jn 48
forders 48
tpch_parent 54
DATA_ROOT 80
flineitem 48
orders 48
load_lineitem 144
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_q04/checkpoints/post_cell_2_try_4.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['date_end', 'flineitem', 'forders', 'keys', 'total', 'jn', 'date_start']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q04/opt_cell_exec_info_2_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[2], f)


In [8]:
opt_output = Out.get(4)

In [9]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q04/annotated/checkpoints/post_cell_2.pickle
assert compare_df(total_opt, total)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['pd']
me:  0
trying: ['load_lineitem']
me:  1
trying: ['date2']
me:  3
trying: ['lineitem']


me:  1
trying: ['lsel']
me:  3
trying: ['orders']


me:  2
trying: ['flineitem']


me:  3
trying: ['load_orders']
me:  2
trying: ['STORAGE_OPTS']
me:  0
trying: ['jn']
me:  3
trying: ['forders']
me:  3
trying: ['orig_output']
me:  4
trying: ['date1']
me:  3
trying: ['total']
me:  3
trying: ['osel']
me:  3
trying: ['tpch_parent']
me:  0
trying: ['DATA_ROOT']
me:  0


Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable orders
Declaring variable load_orders
Declaring variable date2
Declaring variable lsel
Declaring variable flineitem
Declaring variable jn
Declaring variable forders
Declaring variable date1
Declaring variable total
Declaring variable osel
Declaring variable orig_output
